# Named Entity Recognition with spaCy and GLiNER

# Named Entity Recognition with spaCy

In this first part of the notebook, we'll use a standard spaCy framework to extract named entities from a text. spaCy includes access to several language models trained for this task, including transformer-based ones. Next, we'll use displaCy to visualize the results as highlighted annotations with entity labels.  

In the second part, we will use a lightweight transformer-based model, GLiNER, to extract entities with custom labels.

### Variations and Alternatives

#### Different spaCy Models
- **en_core_web_md**: Medium model with better accuracy
- **en_core_web_lg**: Large model with highest accuracy
- **en_core_web_trf**: Transformer-based model with state-of-the-art performance

#### Alternative Libraries
- **NLTK**: Offers basic NER capabilities with different models
- **Hugging Face Transformers**: Provides pre-trained transformer models for NER
- **Stanza**: Stanford's NLP library with robust NER capabilities
- **GLiNER**: lightweight generalist NER model that can be fine-tuned locally and allows custom entity labels.

#### Custom Entity Recognition
- Train custom spaCy models for domain-specific entities
- Use rule-based matching for specific patterns
- Combine multiple models for improved coverage

#### Output Formats
- **BILOU tagging**: More detailed than IOB (Begin, Inside, Last, Outside, Unit)
- **JSON format**: Structured output for API integration
- **CoNLL format**: Standard format for NLP competitions and research

#### Performance Considerations
- For large texts, process in batches using `nlp.pipe()`
- Disable unused pipeline components to improve speed
- Use GPU acceleration for transformer models

## Step 1: Install the necessary libraries
In this step, we prepare the groundwork by installing the necessary frameworks (e.g. spaCy), and libraries (e.g. pandas) for the rest of our notebook.

In [1]:
# install spaCy
%pip install spacy pandas

In [2]:
# install the necessary libraries to parse the text and visualize the results
import spacy
from spacy import displacy
from collections import Counter # we import this module to enable counting entities
import pandas as pd # we import pandas to analyze the data later
pd.options.display.max_rows = 600
pd.options.display.max_colwidth = 400

In [3]:
# download the spacy English language model to process the text
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 63.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
# load the language model
import en_core_web_sm
nlp = en_core_web_sm.load()

## Step 2: Prepare the data
In our previous lesson, we discussed how to clean input data to minimize noise in the results: cleaning double spaces and special characters, deleting numbers outside of the text, and so on. So our text is ready to be processed and we just have to tokenize it before further processing.

In [ ]:
# upload a sample document
from google.colab import files
uploaded = files.upload()

In [5]:
# read the document with the loaded nlp model
text = open("/content/herodotus_histories_sampleparagraph.txt", encoding='utf-8').read()
document = nlp(text)

# extract the text of the tokenized sentences
sentences = [sent.text for sent in document.sents]
print(sentences)

['§ 1.1 I. THE FIRST BOOK OF THE HISTORIES, CALLED CLIO', 'This is the Showing forth of the Inquiry of Herodotus of Halicarnassos, to the end that neither the deeds of men may be forgotten by lapse of time, nor the works great and marvellous, which have been produced some by Hellenes and some by Barbarians, may lose their renown; and especially that the causes may be remembered for which these waged war with one another.', 'Those of the Persians who have knowledge of history declare that the Phoenicians first began the quarrel.', 'These, they say, came from that which is called the Erythraean Sea to this of ours; and having settled in the land where they continue even now to dwell, set themselves forthwith to make long voyages by sea.', "And conveying merchandise of Egypt and of Assyria they arrived at other places and also at Argos; now Argos was at that time in all points the first of the States within that land which is now called Hellas; — the Phoenicians arrived then at this land 

## Working with very large texts: batch-processing

For very long texts, we may consider batch-processing them. spaCy offers the nlp.pipe() module, which takes an iterable of texts and batch-processes them internally as Doc objects. The output is a list of annotated sentences that can then be visualized and manipulated.  

In [6]:
import spacy
nlp = spacy.load("en_core_web_sm")

long_text = open("/content/xenophon_cyropaedia_longer_example.txt", encoding='utf-8').read()

# to facilitate tokenization into sentences, we'll use nlp() disabling ner for the moment
long_document = nlp(long_text, disable=['ner'])
# alternatively, we can use a simple python split (less accurate but much faster initial processing)
#long_document_sentences = long_text.split('. ')

# extract the text from resulting sentence spans
long_document_sentences = [sent.text for sent in long_document.sents]

# process the sentences using nlp.pipe() with NER enabled
processed_long_document_sentences = list(nlp.pipe(long_document_sentences, batch_size=50))

In [7]:
print(processed_long_document_sentences[10:20])

[At the same time, herds are more intractable to strangers than to their rulers and those who derive profit from them., Men, however, conspire against none sooner than against those whom they see attempting to rule over them. 
, § 1.1.3 Thus, as we meditated on this analogy, we were inclined to conclude that for man, as he is constituted, it is easier to rule over any and all other creatures than to rule over men., But when we reflected that there was one Cyrus, the Persian, who reduced to obedience a vast number of men and cities and nations, we were then compelled to change our opinion and decide that to rule men might be a task neither impossible nor even difficult, if one should only go about it in an intelligent manner., At all events, we know that people obeyed Cyrus willingly, although some of them were distant from him a journey of many days, and others of many months; others, although they had never seen him, and still others who knew well that they never should see him., Neve

## Count, analyze, visualize the resulting entities

With the `displacy` module we can visualize all the entities as annotations within the text, and verify the correctness of the results.

In [8]:
displacy.render(document, style="ent", jupyter=True)

Alternatively, we may get all the entities in our document through the `document.ents` property. Using a simple `for` loop, we can also extract the label attribute of each entity in the list.

In [9]:
for named_entity in document.ents:
  print(named_entity, named_entity.label_)

1.1 CARDINAL
CLIO ORG
the Inquiry of Herodotus of Halicarnassos ORG
Hellenes ORG
Barbarians ORG
Persians NORP
Phoenicians NORP
first ORDINAL
the Erythraean Sea LOC
Egypt GPE
Assyria GPE
Argos ORG
Argos PERSON
first ORDINAL
Hellas PERSON
Phoenicians NORP
Argos NORP
the fifth or sixth day DATE
Hellenes ORG
Io LOC
Inachos GPE
Phoenicians NORP
Io LOC
Egypt GPE


We can also filter the entities classified as places, GPE, or other spatially-significant information, by using a simple `if` statement.

In [10]:
for named_entity in document.ents:
  if named_entity.label_ == "NORP" or named_entity.label_ == "PERSON":
    print(named_entity, named_entity.label_)

Persians NORP
Phoenicians NORP
Argos PERSON
Hellas PERSON
Phoenicians NORP
Argos NORP
Phoenicians NORP


In [11]:
# count the number of specific entities extracted from the document
groups = []

for named_entity in document.ents:
  if named_entity.label_ == "NORP" or named_entity.label_ == "GPE":
    groups.append(named_entity.text)

groups_tally = Counter(groups)

df = pd.DataFrame(groups_tally.most_common(), columns=['Group or Organization', 'Count'])
df

,Group or Organization,Count
0,Phoenicians,3
1,Egypt,2
2,Persians,1
3,Assyria,1
4,Argos,1
5,Inachos,1


## Save the results in various formats

Finally, we will want to save the results in markdown or tabular format for further processing.

In [12]:
def doc_to_markdown(doc):
    """ convenience function to convert a spaCy doc to markdown
    with entities marked as [Entity](LABEL)"""
    markdown_text = ""
    last_idx = 0

    for ent in doc.ents:
        # Add text before the entity
        markdown_text += doc.text[last_idx:ent.start_char]
        # Add entity in markdown format
        markdown_text += f"[{ent.text}]({ent.label_})"
        # Update position
        last_idx = ent.end_char

    # Add remaining text after last entity
    markdown_text += doc.text[last_idx:]

    return markdown_text

In [13]:
# Convert all processed sentences of our large document to markdown
markdown_output = []
for doc in processed_long_document_sentences:
    markdown_output.append(doc_to_markdown(doc))

# Join all sentences with spaces or newlines
final_markdown = " ".join(markdown_output)

# Save to file
with open("annotated_output_largedoc.md", "w", encoding="utf-8") as f:
    f.write(final_markdown)

In [14]:
# convert the smaller annotated document to markdown
markdown_output = doc_to_markdown(document)

# save to file
with open("annotated_output_smalldoc.md", "w", encoding="utf-8") as f:
    f.write(markdown_output)

In [15]:
# print the results from a single text as a pandas dataframe
entities = [(entity.text, entity.label_) for entity in document.ents]
df = pd.DataFrame(entities, columns=['Entity', 'Label'])
df

,Entity,Label
0,1.1,CARDINAL
1,CLIO,ORG
2,the Inquiry of Herodotus of Halicarnassos,ORG
3,Hellenes,ORG
4,Barbarians,ORG
5,Persians,NORP
6,Phoenicians,NORP
7,first,ORDINAL
8,the Erythraean Sea,LOC
9,Egypt,GPE


In [16]:
# Extract entities from batch-processed sentences into a dataframe and save to csv
entities_data = []

for i, doc in enumerate(processed_long_document_sentences):
    for ent in doc.ents:
        entities_data.append({
            'sentence_id': i,
            'entity': ent.text,
            'label': ent.label_,
            'start_char': ent.start_char,
            'end_char': ent.end_char,
            'sentence_text': doc.text
        })

# Create DataFrame
df = pd.DataFrame(entities_data)

# save the results
df.to_csv('entities_output_large.csv', index=False)

# Predicting entities with a transformer model (GLiNER)

In this final part of the notebook, we will briefly test the GliNER model for extracting and classifying named entities.

The [GLiNER tranformer model](https://github.com/urchade/GLiNER) is particularly useful because it is lightweight, can be used with custom entity labels, and it can be trained for specific types of texts. So, it is especially flexible if you don't have access to larger models like GPT, or if you have a very specific research question and want to fine-tune a model.

In [17]:
!pip install gliner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.3/76.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.7 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.1
    Uninstalling tokenizers-0.22.1:
      Successfully uninstalled tokenizers-0.22.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.0
    Uninstalling transformers-4.57.0:
      Successfully uninstalled transformers-4.57.0


In [18]:
from gliner import GLiNER

In [19]:
model = GLiNER.from_pretrained("urchade/gliner_mediumv2.1")
model.eval()
print("Successfully loaded the pretrained GLiNER model. For other available models, see http://huggingface.co/urchade.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/781M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/781M [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Successfully loaded the pretrained GLiNER model. For other available models, see http://huggingface.co/urchade.


## Converting the results for visualization

To visualize the results with displaCy, we need to transform them in the correct format. displaCy takes the output format of spaCy by default, which is a list of dictionaries that contains the text, the entity labels, and the extracted entities as entity spans:

```python
ner_output = {
    "text": "Apple is looking at buying U.K. startup for $1 billion",
    "ents": [
        {"start": 0, "end": 5, "label": "ORG"},      # "Apple"
        {"start": 27, "end": 31, "label": "GPE"},    # "U.K."
        {"start": 44, "end": 54, "label": "MONEY"}   # "$1 billion"
    ],
    "title": "My Test Sentence"  # Optional title
}
```

GLiNER output:
```python
gliner_output = [
  {'start': 76, 'end': 84, 'text': 'Hellenes', 'label': 'ethnic', 'score': 0.589714527130127},
  {'start': 113, 'end': 128, 'text': 'barbarian world', 'label': 'location', 'score': 0.8725146651268005}
  ]
```
Below, you'll find a convenience function that will process our GLiNER output and convert it into a format that can be processed by displaCy.

In [20]:
import spacy
from spacy.tokens import Doc, Span

def gliner_output_to_spacy_doc(text, entities, nlp=None):
    """
    Converts GLiNER output (list of dictionaries) into a spaCy Doc with entity spans.

    Args:
        text (str): The original text.
        entities (list): List of dictionaries from GLiNER, e.g., [{'start': 0, 'end': 4, 'text': 'John', 'label': 'PERSON'}].
        nlp (spacy.Language, optional): If None, uses the default English blank model.

    Returns:
        spacy.tokens.Doc: spaCy document with entity spans set.
    """
    if nlp is None:
        nlp = spacy.blank("en")

    # Create a spaCy Doc from the text
    doc = nlp(text)

    # Create Span objects from the GLiNER entities
    spans = []
    for entity in entities:
        try:
            # Find the token indices corresponding to the character start and end
            span = doc.char_span(entity['start'], entity['end'], label=entity['label'])
            if span:
                spans.append(span)
            else:
                print(f"Warning: Could not create span for entity: {entity}. Character indices might not align with tokens.")
        except Exception as e:
            print(f"Error creating span for entity {entity}: {e}")

    # Add the spans to the Doc object
    # Use 'ents' for the standard entity type
    doc.ents = spans

    return doc

## Processing the text with GLiNER and visualizing the results

Next, we process our input text with the GLiNER model, defining the entity labels we want to use.

In [22]:
longer_file = "/content/xenophon_cyropaedia_longer_example.txt"

labels = ["author", "location", "group", "work", "date", "ethnic", "person"]

# open the input file and read the contents
with open(longer_file, "r") as f:
  text_content = f.read() # Read the file content into a string

entities = model.predict_entities(text_content, labels, threshold=0.4) # Pass the string to predict_entities

# print entities and their labels
for entity in entities:
  print(entity["text"], "=>", entity["label"])

# or print the GLiNER output format for all entities
# print(entities)

/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:351: UserWarning: Sentence of length 829 has been truncated to 384
  warnings.warn(f"Sentence of length {len(tokens)} has been truncated to {max_len}")
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


republics => location
people => group
people => group
men => group
private homes => location
people => group
cowherds => group
grooms => person
men => group
herds => group
keeper => person
Men => group


Now we can visualize our entities with displaCy. Since displaCy only takes spaCy entity labels by default, we can customize our own entity labels from the GLiNER output using the `options` parameter, and assign the colors we want.

In [23]:
# Convert GLiNER output to spaCy Doc
doc = gliner_output_to_spacy_doc(text_content, entities)

# using the "options" parameter, we can customize the color of the entities in displacy
colors = {
    "author": "#85C1E2",
    "location": "#FFB6C1",
    "group": "#98FB98"
}

options = {
    "ents": ["author", "location", "group"],
    "colors": colors
}

# render the processed spaCy Doc with GLiNER entities with displaCy
displacy.render(doc, style="ent", jupyter=True, options=options)

# alternatively, we can just render the basic spaCy Doc
#displacy.render(doc, style="ent", jupyter=True)

# another way to display entities and labels from the converted doc, useful for further processing:
# for entity in doc.ents:
#   print(entity, entity.label_)

In [24]:
# print the results as a pandas dataframe
entities = [(entity.text, entity.label_) for entity in doc.ents]
df = pd.DataFrame(entities, columns=['Entity', 'Label'])
df

,Entity,Label
0,republics,location
1,people,group
2,people,group
3,men,group
4,private homes,location
5,people,group
6,cowherds,group
7,grooms,person
8,men,group
9,herds,group
